In [ ]:
!pip install -U datasets pyarrow
!pip install -q ijson
!pip install bert-score

# NOTE: You MUST restart the runtime (Runtime -> Restart session) after this cell finishes,
# otherwise the old versions of datasets and pyarrow will still be loaded in memory.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 28.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.2 MB/s eta 0:00:00


In [ ]:
import os

from datasets import load_dataset


HF_DATASET_ID = "Exploration-Lab/CS-781-Capstone"
hf_token = "API-KEY"


def load_legal_dataset(
    hf_token=None,
    split=None,
    phase=None,
    streaming=False
):

    if hf_token is None:
        raise ValueError(
            "Hugging Face token not found. "
            "Set HF_TOKEN or pass token explicitly."
        )

    dataset = load_dataset(
        HF_DATASET_ID,
        phase,
        split=split,
        token=hf_token,
        streaming=streaming,
    )

    return dataset


In [ ]:
from huggingface_hub import hf_hub_download
import os
from huggingface_hub import login

login()

repo_id = "Exploration-Lab/CS-781-Capstone"

acts_path = hf_hub_download(
    repo_id=repo_id,
    filename="Phase_1_Pretraining/Train/train_acts.json",
    repo_type="dataset",
)

high_courts_path = hf_hub_download(
    repo_id=repo_id,
    filename="Phase_1_Pretraining/Train/train_high_courts.json",
    repo_type="dataset",
)

supreme_court_path = hf_hub_download(
    repo_id=repo_id,
    filename="Phase_1_Pretraining/Train/train_supreme_court.json",
    repo_type="dataset",
)

tribunals_path = hf_hub_download(
    repo_id=repo_id,
    filename="Phase_1_Pretraining/Train/train_tribunals.json",
    repo_type="dataset",
)

print(acts_path)
print(high_courts_path)
print(supreme_court_path)
print(tribunals_path)

Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_supreme_(…): reconstructing file:   0%|          |  0.00B / 1.32GB            

Phase_1_Pretraining/Train/train_supreme_(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json


In [ ]:
import ijson

path = acts_path

with open(path, "rb") as f:
    first = next(ijson.items(f, "item"))

print(type(first))
print(first.keys() if isinstance(first, dict) else first)

<class 'dict'>
dict_keys(['title', 'category', 'year', 'url', 'text', 'word_count', 'text_length_chars'])


In [ ]:
import ijson
import random
import json
import os


def reservoir_sample_json(
    path,
    n_samples,
    seed=42,
):
    rng = random.Random(seed)

    reservoir = []
    seen = 0

    with open(path, "rb") as f:

        for record in ijson.items(f, "item"):

            if not isinstance(record, dict):
                continue

            text = record.get("text")

            if not isinstance(text, str):
                continue

            seen += 1

            if len(reservoir) < n_samples:
                reservoir.append(text)

            else:
                j = rng.randint(0, seen - 1)

                if j < n_samples:
                    reservoir[j] = text

    print(
        f"{os.path.basename(path)}: "
        f"scanned {seen:,} documents, "
        f"selected {len(reservoir):,}"
    )

    return reservoir

In [ ]:
acts = reservoir_sample_json(
    acts_path,
    n_samples=25_000,
    seed=42,
)

high_courts = reservoir_sample_json(
    high_courts_path,
    n_samples=250_000,
    seed=43,
)

supreme_court = reservoir_sample_json(
    supreme_court_path,
    n_samples=25_000,
    seed=44,
)

tribunals = reservoir_sample_json(
    tribunals_path,
    n_samples=80_000,
    seed=45,
)

In [ ]:
rng = random.Random(42)

sampled_texts = (
    acts
    + high_courts
    + supreme_court
    + tribunals
)

rng.shuffle(sampled_texts)

print(f"Total documents: {len(sampled_texts):,}")

In [ ]:
import os
from google.colab import drive
from datasets import load_from_disk, Dataset, DatasetDict

# Mount Google Drive
drive.mount('/content/drive')

# Define the Drive path as requested
cache_dir = '/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache'
os.makedirs(cache_dir, exist_ok=True)


iterable_dataset = load_legal_dataset(
    hf_token=hf_token,
    phase="phase_1",
    streaming=True
)


import os
import psutil
from datasets import IterableDataset

process = psutil.Process(os.getpid())


Mounted at /content/drive


README.md:   0%|          | 0.00/16.1k [00:00<?, ?B/s]

In [ ]:
sample_path = os.path.join(
    cache_dir,
    "bpe_training_sample.json",
)

with open(sample_path, "w", encoding="utf-8") as f:

    for text in sampled_texts:

        f.write(
            json.dumps(
                {"text": text},
                ensure_ascii=False,
            )
            + "\n"
        )

print(f"Saved to: {sample_path}")


In [ ]:
def local_text_stream(path):

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            record = json.loads(line)

            text = record.get("text")

            if isinstance(text, str):
                yield text

In [ ]:
import os
import re
import json
import heapq
import collections

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except Exception:                       # psutil optional
    _PROC = None


def _ram_gb():
    return _PROC.memory_info().rss / (1024 ** 3) if _PROC else 0.0


# ============================================================
# MUST BE IDENTICAL FOR TRAINING AND INFERENCE
# ============================================================
WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)


# ---- multiprocessing helpers (module level so they can be pickled) ----
_WORKER = None

def _init_worker(tok):
    global _WORKER
    _WORKER = tok

def _encode_chunk(args):
    texts, add_special = args
    return [_WORKER.encode(t, add_special) for t in texts]


class BPETokenizer:

    _HEAP_THRESHOLD = 24        # words longer than this use the heap merger
    _CACHE_LIMIT = 500_000

    # ========================================================
    # INIT
    # ========================================================
    def __init__(self, vocab_size, end_marker="</w>", reserved_tokens=None):
        self.vocab_size = vocab_size
        self.end_marker = end_marker
        self.reserved_tokens = (
            reserved_tokens
            if reserved_tokens is not None
            else ["<pad>", "<unk>", "<s>", "</s>"]
        )

        self.itos = list(self.reserved_tokens)
        self.stoi = {t: i for i, t in enumerate(self.reserved_tokens)}

        # (id_a, id_b) -> merge rank. Rank r created token first_merge_id + r.
        self.merges = {}

        self._ready = False
        self._cache = {}

    # ========================================================
    # SAVE / LOAD
    # ========================================================
    def save(self, path):
        data = {
            "vocab_size": self.vocab_size,
            "end_marker": self.end_marker,
            "reserved_tokens": self.reserved_tokens,
            "itos": self.itos,
            "stoi": self.stoi,
            "merges": [[list(p), r] for p, r in self.merges.items()],
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)   # no indent: much smaller/faster

        print(f"Tokenizer saved to: {path}")
        print(f"Vocabulary size: {len(self.itos):,}")
        print(f"Number of merges: {len(self.merges):,}")

    @staticmethod
    def from_json(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        tok = BPETokenizer(
            vocab_size=data["vocab_size"],
            end_marker=data.get("end_marker", "</w>"),
            reserved_tokens=data.get(
                "reserved_tokens", ["<pad>", "<unk>", "<s>", "</s>"]
            ),
        )
        tok.itos = data["itos"]
        tok.stoi = data["stoi"]
        tok.merges = {tuple(p): r for p, r in data["merges"]}
        tok._ready = False
        tok._cache = {}
        return tok

    # ========================================================
    # BASE VOCAB
    # ========================================================
    def _initialize_bytes(self):
        """Reserved tokens, then 256 byte tokens, then </w>."""
        for b in range(256):
            tok = bytes([b]).decode("latin-1")
            if tok not in self.stoi:
                self.stoi[tok] = len(self.itos)
                self.itos.append(tok)

        if self.end_marker not in self.stoi:
            self.stoi[self.end_marker] = len(self.itos)
            self.itos.append(self.end_marker)

    # ========================================================
    # WORD FREQUENCIES  (memory-safe)
    # ========================================================
    def _get_initial_vocab(
        self,
        corpus_texts,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        """
        Count words as plain strings (far lighter than tuples of ints),
        periodically dropping the long tail (typos, ids, citation numbers),
        with a hard cap on the number of unique words. Only the survivors are
        converted to byte-id tuples.

        Dropped rare words are still encodable at inference through byte
        fallback, so this costs almost nothing in tokenizer quality.
        """
        self._initialize_bytes()
        end_id = self.stoi[self.end_marker]

        counts = collections.Counter()
        threshold = prune_threshold
        n = 0
        last_ram = _ram_gb()

        def prune(th):
            return collections.Counter(
                {w: c for w, c in counts.items() if c >= th}
            )

        for n, text in enumerate(corpus_texts, 1):
            if not isinstance(text, str):
                continue

            if len(text) > max_doc_chars:
                print(
                    f"  [warn] example {n:,} has {len(text):,} chars "
                    f"-> truncated to {max_doc_chars:,}"
                )
                text = text[:max_doc_chars]

            # generator: never builds a giant list of words in memory
            counts.update(m.group() for m in WORD_RE.finditer(text))

            if n % 100 == 0:
                ram = _ram_gb()
                if ram - last_ram > 0.3:
                    print(f"  [warn] RAM jumped {last_ram:.2f} -> {ram:.2f} GB near example {n:,}")
                last_ram = ram

            if n % prune_every == 0:
                counts = prune(threshold)

            # hard safety cap: raise threshold until we fit
            while len(counts) > max_unique_words:
                threshold += 1
                counts = prune(threshold)
                print(f"  cap hit -> prune threshold raised to {threshold}")

            if n % 1000 == 0:
                print(
                    f"Examples: {n:,} | Unique words: {len(counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if n >= max_examples:
                break

        counts = prune(prune_threshold)     # final prune

        # convert survivors to id tuples
        byte_ids = [self.stoi[bytes([b]).decode("latin-1")] for b in range(256)]
        word_freqs = {}
        for w, c in counts.items():
            ids = tuple(byte_ids[b] for b in w.encode("utf-8")) + (end_id,)
            word_freqs[ids] = word_freqs.get(ids, 0) + c
        del counts

        print(f"\nFinished reading corpus. Examples: {n:,}")
        print(f"Unique words kept: {len(word_freqs):,}")
        return word_freqs

    # ========================================================
    # TRAIN
    # ========================================================
    def train(
        self,
        corpus_texts,
        save_checkpoints=None,
        checkpoint_dir=None,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        save_checkpoints = set(save_checkpoints or [])
        checkpoint_dir = checkpoint_dir or "."
        os.makedirs(checkpoint_dir, exist_ok=True)

        # ---- Step 1: word frequencies --------------------------------
        print("\nSTEP 1: word frequencies")
        word_freqs = self._get_initial_vocab(
            corpus_texts,
            max_examples=max_examples,
            prune_every=prune_every,
            prune_threshold=prune_threshold,
            max_unique_words=max_unique_words,
            max_doc_chars=max_doc_chars,
        )

        words = [list(w) for w in word_freqs]
        freqs = list(word_freqs.values())
        del word_freqs

        # ---- Step 2: pair statistics ---------------------------------
        print("\nSTEP 2: pair statistics")
        pair_counts = collections.defaultdict(int)
        pair_to_words = collections.defaultdict(set)     # pair -> word indices

        for idx, (w, f) in enumerate(zip(words, freqs)):
            for pair in zip(w, w[1:]):
                pair_counts[pair] += f
                pair_to_words[pair].add(idx)

        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        print(f"Initial vocab size: {len(self.itos):,}")
        print(f"Unique words:       {len(words):,}")
        print(f"Unique pairs:       {len(pair_counts):,}")
        print(f"RAM: {_ram_gb():.2f} GB")

        # ---- Step 3: merges ------------------------------------------
        print("\nSTEP 3: BPE merges")

        while len(self.itos) < self.vocab_size:

            # pop until an up-to-date entry is found (lazy deletion)
            best_pair, best_freq = None, 0
            while heap:
                neg, pair = heapq.heappop(heap)
                if -neg > 0 and pair_counts.get(pair, 0) == -neg:
                    best_pair, best_freq = pair, -neg
                    break

            if best_pair is None:
                print("No more pairs available.")
                break

            a, b = best_pair

            new_id = len(self.itos)
            new_str = self.itos[a] + self.itos[b]
            self.itos.append(new_str)
            self.stoi.setdefault(new_str, new_id)
            self.merges[best_pair] = len(self.merges)

            affected = pair_to_words.pop(best_pair, set())
            changed = set()

            for idx in affected:
                w, f = words[idx], freqs[idx]

                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                    if p != best_pair:
                        s = pair_to_words.get(p)
                        if s is not None:
                            s.discard(idx)

                out, i, n = [], 0, len(w)
                while i < n:
                    if i < n - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                    else:
                        out.append(w[i])
                        i += 1
                words[idx] = out

                for p in zip(out, out[1:]):
                    pair_counts[p] += f
                    pair_to_words[p].add(idx)
                    changed.add(p)

            pair_counts.pop(best_pair, None)
            changed.discard(best_pair)

            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_to_words.pop(p, None)

            # rebuild heap if it is mostly stale (bounds heap memory)
            if len(heap) > 4 * len(pair_counts) + 10_000:
                heap = [(-c, p) for p, c in pair_counts.items()]
                heapq.heapify(heap)

            vocab_now = len(self.itos)

            if vocab_now % 1000 == 0:
                print(
                    f"Vocab: {vocab_now:,} | merge freq: {best_freq:,} | "
                    f"affected: {len(affected):,} | pairs: {len(pair_counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if vocab_now in save_checkpoints:
                self.save(os.path.join(checkpoint_dir, f"bpe_{vocab_now}-large.json"))

        self._ready = False
        self._cache = {}

        print("\nBPE training complete.")
        print(f"Final vocabulary size: {len(self.itos):,}")
        print(f"Merges learned:        {len(self.merges):,}")

    # ========================================================
    # INFERENCE SETUP
    # ========================================================
    def _prepare_inference(self):
        if self._ready:
            return

        self._byte_ids = [
            self.stoi[bytes([b]).decode("latin-1")] for b in range(256)
        ]
        self._first_merge_id = len(self.itos) - len(self.merges)
        self._end_id = self.stoi.get(self.end_marker, self.stoi["<unk>"])
        self._bos = self.stoi["<s>"]
        self._eos = self.stoi["</s>"]

        # sanity check: rank r must have created token first_merge_id + r
        base = self._first_merge_id
        for (a, b), r in self.merges.items():
            if self.itos[base + r] != self.itos[a] + self.itos[b]:
                raise ValueError(
                    "Merge/vocab mismatch: cannot infer merge ids from ranks."
                )

        # integer keys: faster than tuple keys
        self._M = len(self.itos)
        M = self._M
        self._ranks = {a * M + b: r for (a, b), r in self.merges.items()}

        self._cache = {}
        self._ready = True

    # ========================================================
    # MERGE STRATEGIES
    # ========================================================
    def _merge_small(self, ids):
        """Repeated best-rank scan. Fastest for short words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        while len(ids) > 1:
            best, best_i = None, -1
            for i in range(len(ids) - 1):
                r = get(ids[i] * M + ids[i + 1])
                if r is not None and (best is None or r < best):
                    best, best_i = r, i

            if best is None:
                break

            a, b, new = ids[best_i], ids[best_i + 1], base + best
            out, i, n = [], 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out

        return ids

    def _merge_heap(self, ids):
        """Linked list + min-heap: O(n log n) for long words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        n = len(ids)
        nxt = list(range(1, n + 1))
        nxt[-1] = -1
        prv = list(range(-1, n - 1))
        alive = [True] * n

        heap = []
        for i in range(n - 1):
            r = get(ids[i] * M + ids[i + 1])
            if r is not None:
                heap.append((r, i))
        heapq.heapify(heap)

        while heap:
            r, i = heapq.heappop(heap)
            if not alive[i]:
                continue
            j = nxt[i]
            if j == -1 or get(ids[i] * M + ids[j]) != r:
                continue                        # stale entry

            ids[i] = base + r
            alive[j] = False
            k = nxt[j]
            nxt[i] = k
            if k != -1:
                prv[k] = i

            p = prv[i]
            if p != -1:
                r2 = get(ids[p] * M + ids[i])
                if r2 is not None:
                    heapq.heappush(heap, (r2, p))
            if k != -1:
                r2 = get(ids[i] * M + ids[k])
                if r2 is not None:
                    heapq.heappush(heap, (r2, i))

        return [ids[i] for i in range(n) if alive[i]]

    def _bpe_word(self, word):
        hit = self._cache.get(word)
        if hit is not None:
            return hit

        ids = [self._byte_ids[b] for b in word.encode("utf-8")]
        ids.append(self._end_id)

        if len(ids) <= self._HEAP_THRESHOLD:
            ids = self._merge_small(ids)
        else:
            ids = self._merge_heap(ids)

        result = tuple(ids)
        if len(self._cache) < self._CACHE_LIMIT:
            self._cache[word] = result
        return result

    # ========================================================
    # PUBLIC API
    # ========================================================
    def encode(self, text, add_special_tokens=True):
        """Fast path: token ids only."""
        self._prepare_inference()

        out = [self._bos] if add_special_tokens else []
        extend = out.extend
        cache_get = self._cache.get
        bpe = self._bpe_word

        for w in WORD_RE.findall(text):
            t = cache_get(w)
            if t is None:
                t = bpe(w)
            extend(t)

        if add_special_tokens:
            out.append(self._eos)
        return out

    def tokenize(self, text, add_special_tokens=True):
        """Returns (token_ids, token_strings). Use encode() for bulk work."""
        ids = self.encode(text, add_special_tokens)
        itos = self.itos
        return ids, [itos[i] for i in ids]

    def encode_batch(self, texts, add_special_tokens=True,
                     num_workers=None, chunk_size=256):
        """Encode many documents, optionally across processes (fork only)."""
        self._prepare_inference()
        texts = list(texts)

        if num_workers is None:
            num_workers = os.cpu_count() or 1

        if num_workers <= 1 or len(texts) < 2 * chunk_size:
            return [self.encode(t, add_special_tokens) for t in texts]

        import multiprocessing as mp
        chunks = [
            (texts[i:i + chunk_size], add_special_tokens)
            for i in range(0, len(texts), chunk_size)
        ]
        ctx = mp.get_context("fork")
        with ctx.Pool(num_workers, initializer=_init_worker,
                      initargs=(self,)) as pool:
            results = list(pool.imap(_encode_chunk, chunks))
        return [ids for chunk in results for ids in chunk]

    def decode(self, token_ids):
        """Decodes a list of token IDs back into a string, handling special tokens."""
        self._prepare_inference()
        # Filter out special tokens for a cleaner decoded string
        filtered_ids = [
            tid for tid in token_ids
            if self.itos[tid] not in self.reserved_tokens
        ]
        # Join the token strings, replacing </w> with a space
        decoded_text = "".join([self.itos[tid] for tid in filtered_ids]).replace(self.end_marker, " ")
        return decoded_text.strip()


In [ ]:
vocab_size = 32768
tokenizer = BPETokenizer(vocab_size=vocab_size)

print(f"Training BPE tokenizer with vocab size {vocab_size}...")

tokenizer = BPETokenizer(
    vocab_size=vocab_size,
    reserved_tokens=["<pad>", "<unk>", "<s>", "</s>", "<mask>", "<sep>", "<cls>"],
)
tokenizer.train(
    local_text_stream(sample_path),
    save_checkpoints=[8192, 16384, 32768],
    checkpoint_dir=cache_dir,
    max_examples=25_000,        # more coverage, see the caveat below
    max_doc_chars=200_000,
    prune_every=5_000,
    prune_threshold=2,          # keeps mid-frequency words
    max_unique_words=2_000_000, # about 300 MB, protects RAM
)

print("Tokenizer training finished!")

bpe_en_path = os.path.join(cache_dir, 'final_bpe_en.json')
tokenizer.save(bpe_en_path)


Training BPE tokenizer with vocab size 65536...

STEP 1: word frequencies
Examples: 1,000 | Unique words: 38,243 | RAM: 34.74 GB
  [warn] example 1,101 has 224,780 chars -> truncated to 200,000
  [warn] example 1,117 has 212,349 chars -> truncated to 200,000
  [warn] example 1,413 has 204,145 chars -> truncated to 200,000
  [warn] example 1,731 has 233,584 chars -> truncated to 200,000
Examples: 2,000 | Unique words: 56,280 | RAM: 34.74 GB
  [warn] example 2,043 has 233,216 chars -> truncated to 200,000
Examples: 3,000 | Unique words: 71,050 | RAM: 34.74 GB
  [warn] example 3,568 has 672,618 chars -> truncated to 200,000
  [warn] example 3,716 has 200,471 chars -> truncated to 200,000
  [warn] example 3,808 has 213,103 chars -> truncated to 200,000
Examples: 4,000 | Unique words: 84,897 | RAM: 34.74 GB
  [warn] example 4,819 has 241,619 chars -> truncated to 200,000
  [warn] example 4,957 has 318,723 chars -> truncated to 200,000
Examples: 5,000 | Unique words: 53,411 | RAM: 34.74 GB
 

In [ ]:
# ============================================================
# BPE TOKENIZER TEST
# ============================================================
_VOCAB_SIZE = 16384
bpe_en_path = os.path.join(cache_dir, f'bpe_{_VOCAB_SIZE}.json')
tokenizer = BPETokenizer.from_json(bpe_en_path)

print("Tokenizer loaded successfully")
print(f"Vocabulary size: {len(tokenizer.itos):,}")
print(f"Number of merges: {len(tokenizer.merges):,}")

test_texts = [
    "The Supreme Court of India held that the right to equality is a fundamental right.",

    "Section 302 of the Indian Penal Code provides punishment for murder.",

    "The petitioner challenged the constitutional validity of the impugned provision.",

    "Section 21: Protection of life and personal liberty shall not be denied.",
]

for text in test_texts:

    token_ids, token_strings = tokenizer.tokenize(
        text,
        add_special_tokens=True,
    )

    print("\n" + "=" * 80)
    print("TEXT:")
    print(text)

    print("\nTOKEN IDS:")
    print(token_ids)

    print("\nTOKENS:")
    print(token_strings)

    print("\nSTATISTICS:")
    print(f"Characters : {len(text):,}")
    print(f"Tokens     : {len(token_ids):,}")
    print(
        f"Chars/token: "
        f"{len(text) / max(len(token_ids), 1):.2f}"
    )

Tokenizer loaded successfully
Vocabulary size: 16,384
Number of merges: 16,120

TEXT:
The Supreme Court of India held that the right to equality is a fundamental right.

TOKEN IDS:
[2, 391, 4995, 690, 278, 899, 1448, 413, 271, 1585, 295, 8030, 320, 307, 15849, 1585, 266, 3]

TOKENS:
['<s>', 'The</w>', 'Supreme</w>', 'Court</w>', 'of</w>', 'India</w>', 'held</w>', 'that</w>', 'the</w>', 'right</w>', 'to</w>', 'equality</w>', 'is</w>', 'a</w>', 'fundamental</w>', 'right</w>', '.</w>', '</s>']

STATISTICS:
Characters : 82
Tokens     : 18
Chars/token: 4.56

TEXT:
Section 302 of the Indian Penal Code provides punishment for murder.

TOKEN IDS:
[2, 637, 13567, 278, 271, 1414, 4361, 1430, 6664, 3399, 329, 11302, 266, 3]

TOKENS:
['<s>', 'Section</w>', '302</w>', 'of</w>', 'the</w>', 'Indian</w>', 'Penal</w>', 'Code</w>', 'provides</w>', 'punishment</w>', 'for</w>', 'murder</w>', '.</w>', '</s>']

STATISTICS:
Characters : 68
Tokens     : 14
Chars/token: 4.86

TEXT:
The petitioner challenged th

In [ ]:
text = "reiterated by this Court observing that acquisition proceedings should be challenged before the same attain finality , in State of Mysore Vs . V . K . Kangan AIR 1975 SC 2190 ; [ PT . Girdharan Prasad Missir Vs . State of Bihar ] ( 1980 ) 2 SCC 83 ; Bhoop Singh Vs . Union of India AIR 1992 SC 1414 ; State of Orissa Vs . Dhobei Sethi & Anr . ( 1995 ) 5 SCC 583 ; State of Maharashtra Vs . Digambar AIR 1995 SC 1991 ; State of Tamil Nadu Vs . L . Krishnan AIR 1996 SC 497 ; and [ C . Padma & Ors . Vs . Dy . Secretary to Govt . of Tamil Nadu & Ors . ] ( 1997 ) 2 SCC 627 . 9 . In [ Municipal Corporation of Greater Bombay Vs . Industrial Development Investment Co . Pvt . Ltd . & Ors . ] AIR 1997 SC 482 , this Court observed as under : - \" If the interested person allows the grass to grow under his feet by allowing the acquisition proceedings to go on and reach its terminus in the award and possession is taken in furtherance thereof and vest in the State free from all encumbrances , the slumbered interested person would be told off the gates of the Court"

ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print("ORIGINAL:")
print(text)

print("\nIDS:")
print(ids)

print("\nDECODED:")
print(decoded)

print("\nMATCH:")
print(text == decoded)

ORIGINAL:
reiterated by this Court observing that acquisition proceedings should be challenged before the same attain finality , in State of Mysore Vs . V . K . Kangan AIR 1975 SC 2190 ; [ PT . Girdharan Prasad Missir Vs . State of Bihar ] ( 1980 ) 2 SCC 83 ; Bhoop Singh Vs . Union of India AIR 1992 SC 1414 ; State of Orissa Vs . Dhobei Sethi & Anr . ( 1995 ) 5 SCC 583 ; State of Maharashtra Vs . Digambar AIR 1995 SC 1991 ; State of Tamil Nadu Vs . L . Krishnan AIR 1996 SC 497 ; and [ C . Padma & Ors . Vs . Dy . Secretary to Govt . of Tamil Nadu & Ors . ] ( 1997 ) 2 SCC 627 . 9 . In [ Municipal Corporation of Greater Bombay Vs . Industrial Development Investment Co . Pvt . Ltd . & Ors . ] AIR 1997 SC 482 , this Court observed as under : - " If the interested person allows the grass to grow under his feet by allowing the acquisition proceedings to go on and reach its terminus in the award and possession is taken in furtherance thereof and vest in the State free from all encumbrances , t

In [ ]:
# ============================================================
# BPE TOKENIZER TEST
# ============================================================

_VOCAB_SIZE = 32768
bpe_en_path = os.path.join(cache_dir, f'bpe_{_VOCAB_SIZE}.json')
tokenizer = BPETokenizer.from_json(bpe_en_path)

print("Tokenizer loaded successfully")
print(f"Vocabulary size: {len(tokenizer.itos):,}")
print(f"Number of merges: {len(tokenizer.merges):,}")

test_texts = [
    "The Supreme Court of India held that the right to equality is a fundamental right.",

    "Section 302 of the Indian Penal Code provides punishment for murder.",

    "The petitioner challenged the constitutional validity of the impugned provision.",

    "Section 21: Protection of life and personal liberty shall not be denied.",
]

for text in test_texts:

    token_ids, token_strings = tokenizer.tokenize(
        text,
        add_special_tokens=True,
    )

    print("\n" + "=" * 80)
    print("TEXT:")
    print(text)

    print("\nTOKEN IDS:")
    print(token_ids)

    print("\nTOKENS:")
    print(token_strings)

    print("\nSTATISTICS:")
    print(f"Characters : {len(text):,}")
    print(f"Tokens     : {len(token_ids):,}")
    print(
        f"Chars/token: "
        f"{len(text) / max(len(token_ids), 1):.2f}"
    )

In [ ]:
acts = reservoir_sample_json(
    acts_path,
    n_samples=20_000,
    seed=42,
)

high_courts = reservoir_sample_json(
    high_courts_path,
    n_samples=300_000,
    seed=42,
)

supreme_court = reservoir_sample_json(
    supreme_court_path,
    n_samples=50_000,
    seed=42,
)

tribunals = reservoir_sample_json(
    tribunals_path,
    n_samples=200_000,
    seed=42,
)

train_acts.json: scanned 18,826 documents, selected 18,826
train_high_courts.json: scanned 2,536,738 documents, selected 300,000
train_supreme_court.json: scanned 55,549 documents, selected 50,000
train_tribunals.json: scanned 869,132 documents, selected 200,000


In [ ]:
import random
rng = random.Random(42)

train_val_texts = (
    acts
    + high_courts
    + supreme_court
    + tribunals
)

# rng.shuffle(train_val_texts)

print(f"Total documents: {len(train_val_texts):,}")

Total documents: 568,826


## Data cleaning: whitespace, languages, duplicates

Runs on `train_val_texts` **before** the train/val split, so duplicates cannot leak from train into validation.

1. **Normalisation**: curly quotes/dashes become ASCII, NFKC fixes nbsp/ligatures/full-width forms, zero-width and control characters are removed, every run of whitespace becomes one space, and runs like `.......` or `-----` are shortened to 3.
2. **Languages**: documents that are not mostly Latin script, or not recognisably English, are dropped. Stray Hindi/Urdu/etc. fragments inside English documents are stripped.
3. **Duplicates**: exact duplicates (after normalisation) and near-duplicates (MinHash-LSH, ~0.8 Jaccard) are removed.

`normalize_text` is also applied to the test prompts at inference so train and test text match.

In [ ]:
# ============================================================
# DATA CLEANING
#   1. Unicode + whitespace normalisation
#   2. Language / script filtering (keep English, drop the rest)
#   3. De-duplication (exact + near-duplicate via MinHash-LSH)
#
# Run this AFTER the raw documents are sampled (train_val_texts)
# and BEFORE the train / val split, so that duplicates cannot
# leak from train into validation.
# ============================================================
import re
import zlib
import hashlib
import unicodedata
import collections
import numpy as np


# ------------------------------------------------------------
# 1. TEXT NORMALISATION
#    normalize_text() is also applied to the test prompts at
#    inference time so train and test text look the same.
# ------------------------------------------------------------
# characters that are deleted outright (zero-width, bidi marks, soft hyphen,
# BOM, the U+FFFD "decode error" marker)
_CHAR_MAP = {ord(c): None for c in (
    "\u200b\u200c\u200d\u200e\u200f\u2060\ufeff\u00ad\ufffd"
    "\u202a\u202b\u202c\u202d\u202e"
)}
# typographic quotes / dashes -> ASCII, so one symbol is not split over 3 variants
_CHAR_MAP.update({ord(c): "'" for c in "\u2018\u2019\u201a\u201b\u2032\u02bc"})
_CHAR_MAP.update({ord(c): '"' for c in "\u201c\u201d\u201e\u201f\u2033"})
_CHAR_MAP.update({ord(c): "-" for c in "\u2010\u2011\u2012\u2013\u2014\u2015\u2212"})

_CTRL_RE = re.compile(r"[\x00-\x08\x0e-\x1b\x7f\x80-\x84\x86-\x9f]")
_WS_RE = re.compile(r"\s+")                      # any run of unicode whitespace
_REPEAT_RE = re.compile(r"([^\w\s]|_)\1{3,}")    # '......', '-----', '_____' ...


def normalize_text(text, collapse_repeats=True):
    """Unicode-normalise and collapse whitespace.

    - quotes/dashes -> ASCII, NFKC (turns nbsp, thin/ideographic spaces,
      ligatures, full-width forms into plain characters)
    - removes zero-width / control / replacement characters
    - every run of whitespace (spaces, tabs, newlines, nbsp ...) -> one ' '
    - runs of 4+ identical punctuation chars ('.....', '-----') -> 3
    """
    text = text.translate(_CHAR_MAP)
    if not text.isascii():
        text = unicodedata.normalize("NFKC", text)
    text = _CTRL_RE.sub("", text)
    text = _WS_RE.sub(" ", text).strip()
    if collapse_repeats:
        text = _REPEAT_RE.sub(lambda m: m.group(1) * 3, text)
    return text


# ------------------------------------------------------------
# 2. LANGUAGE / SCRIPT HANDLING (no external dependency)
# ------------------------------------------------------------
_LATIN = "A-Za-z\u00C0-\u024F\u1E00-\u1EFF"
_SCRIPTS = {
    "devanagari": "\u0900-\u097F", "bengali": "\u0980-\u09FF",
    "gurmukhi":   "\u0A00-\u0A7F", "gujarati": "\u0A80-\u0AFF",
    "odia":       "\u0B00-\u0B7F", "tamil":    "\u0B80-\u0BFF",
    "telugu":     "\u0C00-\u0C7F", "kannada":  "\u0C80-\u0CFF",
    "malayalam":  "\u0D00-\u0D7F",
    "arabic/urdu": "\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF",
    "cyrillic":   "\u0400-\u04FF",
    "cjk":        "\u3040-\u30FF\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF",
}
_SCRIPT_RES = {k: re.compile(f"[{v}]") for k, v in _SCRIPTS.items()}
_ALPHA_RE = re.compile(r"[^\W\d_]")
_LATIN_RE = re.compile(f"[{_LATIN}]")
# a run of non-Latin letters (plus the combining marks / joiners that belong to them)
_FOREIGN_RUN_RE = re.compile(
    "[" + "".join(_SCRIPTS.values()) + "]+"
    "(?:[\\s\u200c\u200d]*[" + "".join(_SCRIPTS.values()) + "]+)*"
)
_EN_WORD_RE = re.compile(r"[a-z]+")
_ENGLISH_STOP = frozenset("""
the of and to in that is was for by on with as be it at this which are from or not has have
had been an a he she they his her their its were any such under also but if all no can shall
may would
""".split())


def language_profile(text, sample_chars=3000):
    """Cheap profile from the start and middle of a document.
    Returns (latin_ratio, english_stopword_ratio, dominant_foreign_script)."""
    n = len(text)
    s = text if n <= 2 * sample_chars else (
        text[:sample_chars] + " " + text[n // 2: n // 2 + sample_chars])
    n_alpha = len(_ALPHA_RE.findall(s))
    if n_alpha == 0:
        return 0.0, 0.0, "none"
    n_latin = len(_LATIN_RE.findall(s))
    latin_ratio = n_latin / n_alpha

    words = _EN_WORD_RE.findall(s.lower())
    en_ratio = (sum(w in _ENGLISH_STOP for w in words) / len(words)) if words else 0.0

    foreign = "latin-non-english"
    if latin_ratio < 0.5:
        counts = {k: len(r.findall(s)) for k, r in _SCRIPT_RES.items()}
        best = max(counts, key=counts.get)
        foreign = best if counts[best] > 0 else "other"
    return latin_ratio, en_ratio, foreign


# ------------------------------------------------------------
# 3. NEAR-DUPLICATE DETECTION (MinHash + LSH banding)
# ------------------------------------------------------------
_WORD_RE = re.compile(r"\w+")
_MH_PRIME = np.uint64(4294967311)           # prime just above 2**32
_ROLL = np.uint64(1000003)


class NearDupIndex:
    """Streaming near-duplicate detector. The first copy seen is kept."""

    def __init__(self, num_perm=64, bands=8, threshold=0.8,
                 shingle=5, max_words=2000, seed=1):
        assert num_perm % bands == 0
        self.num_perm, self.bands = num_perm, bands
        self.rows = num_perm // bands
        self.threshold, self.shingle, self.max_words = threshold, shingle, max_words
        rng = np.random.RandomState(seed)
        self.a = rng.randint(1, 2 ** 31 - 1, size=num_perm).astype(np.uint64)[:, None]
        self.b = rng.randint(0, 2 ** 31 - 1, size=num_perm).astype(np.uint64)[:, None]
        self.sigs = []
        self.buckets = [dict() for _ in range(bands)]

    def _shingle_hashes(self, words):
        words = words[: self.max_words]
        k = self.shingle
        wh = np.fromiter((zlib.crc32(w.encode()) for w in words),
                         dtype=np.uint64, count=len(words))
        m = len(wh) - k + 1
        if m < 1:
            return wh
        sh = wh[:m].copy()
        for j in range(1, k):                 # rolling hash of k consecutive words
            sh = sh * _ROLL + wh[j: m + j]
        return (sh ^ (sh >> np.uint64(32))) & np.uint64(0xFFFFFFFF)

    def check_and_add(self, words):
        """True if a near-duplicate was already indexed; otherwise index and return False."""
        h = self._shingle_hashes(words)[None, :]
        sig = ((self.a * h + self.b) % _MH_PRIME).min(axis=1)
        r = self.rows
        keys = [hash(sig[i * r:(i + 1) * r].tobytes()) for i in range(self.bands)]
        for i, k in enumerate(keys):
            j = self.buckets[i].get(k)
            if j is not None and (self.sigs[j] == sig).mean() >= self.threshold:
                return True
        idx = len(self.sigs)
        self.sigs.append(sig)
        for i, k in enumerate(keys):
            self.buckets[i].setdefault(k, idx)
        return False


# ------------------------------------------------------------
# PIPELINE
# ------------------------------------------------------------
def clean_corpus(
    texts,
    min_words=30,                 # drop near-empty documents
    language_filter=True,         # drop non-English documents
    min_latin_ratio=0.90,         # share of letters that must be Latin script
    min_english_ratio=0.12,       # share of words that are common English words
    strip_foreign_runs=True,      # remove stray Hindi/Urdu/... fragments inside English docs
    exact_dedup=True,
    near_dedup=True,
    near_threshold=0.80,          # estimated Jaccard similarity above which = duplicate
    collapse_repeats=True,
    n_examples=3,
    progress_every=20_000,
):
    stats = collections.Counter()
    dropped_scripts = collections.Counter()
    examples = collections.defaultdict(list)
    seen_exact = set()
    near = NearDupIndex(threshold=near_threshold) if near_dedup else None
    kept = []

    def drop(reason, text):
        stats[reason] += 1
        if len(examples[reason]) < n_examples:
            examples[reason].append(text[:200])

    for n, raw in enumerate(texts, 1):
        stats["input_docs"] += 1
        if not isinstance(raw, str):
            drop("not_a_string", str(raw)); continue

        # 1. normalise -------------------------------------------------
        text = normalize_text(raw, collapse_repeats=collapse_repeats)
        if text != raw:
            stats["docs_changed_by_normalisation"] += 1

        # 2. cheap length pre-check (so tiny docs are reported as "too short",
        #    not mislabelled as non-English)
        if text.count(" ") + 1 < min_words:
            drop("dropped_too_short", text); continue

        # 3. language --------------------------------------------------
        if language_filter:
            latin, en, script = language_profile(text)
            if latin < min_latin_ratio:
                dropped_scripts[script] += 1
                drop("dropped_non_latin_script", text); continue
            if en < min_english_ratio:
                dropped_scripts["latin-non-english"] += 1
                drop("dropped_non_english_latin", text); continue
            if strip_foreign_runs:
                cleaned = _FOREIGN_RUN_RE.sub(" ", text)
                if cleaned != text:
                    stats["docs_with_foreign_fragments_stripped"] += 1
                    text = _WS_RE.sub(" ", cleaned).strip()

        # 4. final length check (after foreign fragments were stripped) -
        words = _WORD_RE.findall(text.lower())
        if len(words) < min_words:
            drop("dropped_too_short", text); continue

        # 5. exact duplicates ------------------------------------------
        if exact_dedup:
            key = hashlib.blake2b(" ".join(words).encode(), digest_size=16).digest()
            if key in seen_exact:
                drop("dropped_exact_duplicate", text); continue
            seen_exact.add(key)

        # 6. near duplicates -------------------------------------------
        if near is not None and near.check_and_add(words):
            drop("dropped_near_duplicate", text); continue

        kept.append(text)
        if progress_every and n % progress_every == 0:
            print(f"  processed {n:,} | kept {len(kept):,}")

    stats["kept_docs"] = len(kept)
    return kept, {"stats": stats, "dropped_scripts": dropped_scripts, "examples": examples}


def print_clean_report(report, chars_before=None, chars_after=None):
    s = report["stats"]
    n = max(s["input_docs"], 1)
    print("=" * 62)
    print(f"Input documents                : {s['input_docs']:>9,}")
    for k in ["not_a_string", "dropped_non_latin_script", "dropped_non_english_latin",
              "dropped_too_short", "dropped_exact_duplicate", "dropped_near_duplicate"]:
        if s[k]:
            print(f"  - {k:<28}: {s[k]:>9,}  ({100 * s[k] / n:5.1f}%)")
    print(f"Kept documents                 : {s['kept_docs']:>9,}  ({100 * s['kept_docs'] / n:5.1f}%)")
    print(f"Docs changed by normalisation  : {s['docs_changed_by_normalisation']:>9,}")
    print(f"Docs with foreign text stripped: {s['docs_with_foreign_fragments_stripped']:>9,}")
    if chars_before and chars_after:
        print(f"Characters                     : {chars_before:,} -> {chars_after:,}")
    if report["dropped_scripts"]:
        print("\nLanguages / scripts removed:")
        for k, v in report["dropped_scripts"].most_common():
            print(f"  {k:<20}{v:>9,}")
    for reason, ex in report["examples"].items():
        print(f"\n[{reason}] sample:")
        for e in ex:
            print("   ", repr(e[:140]))
    print("=" * 62)


In [ ]:
chars_before = sum(len(t) for t in train_val_texts)

train_val_texts, clean_report = clean_corpus(
    train_val_texts,
    min_words=30,
    language_filter=True,
    min_latin_ratio=0.90,
    min_english_ratio=0.12,
    strip_foreign_runs=True,
    exact_dedup=True,
    near_dedup=True,
    near_threshold=0.80,
)

chars_after = sum(len(t) for t in train_val_texts)
print_clean_report(clean_report, chars_before, chars_after)

# free the raw per-source lists (they still hold the uncleaned text in RAM)
import gc
del acts, high_courts, supreme_court, tribunals
gc.collect()

rng = random.Random(42)
# rng.shuffle(train_val_texts)
print(f"Total documents after cleaning: {len(train_val_texts):,}")

  processed 20,000 | kept 19,035
  processed 40,000 | kept 38,546
  processed 60,000 | kept 57,767
  processed 80,000 | kept 76,848
  processed 100,000 | kept 95,793
  processed 120,000 | kept 114,672
  processed 140,000 | kept 133,464
  processed 160,000 | kept 152,248
  processed 180,000 | kept 170,886
  processed 200,000 | kept 189,472
  processed 220,000 | kept 208,028
  processed 240,000 | kept 226,656
  processed 260,000 | kept 245,267
  processed 280,000 | kept 263,838
  processed 300,000 | kept 282,247
  processed 320,000 | kept 300,726
  processed 340,000 | kept 320,447
  processed 360,000 | kept 340,120
  processed 400,000 | kept 376,934
  processed 460,000 | kept 430,137
  processed 500,000 | kept 465,393
  processed 520,000 | kept 482,886
  processed 560,000 | kept 518,509
Input documents                :   568,826
  - dropped_non_latin_script    :     6,143  (  1.1%)
  - dropped_non_english_latin   :    11,317  (  2.0%)
  - dropped_too_short           :     2,088  (  0.4%)

In [ ]:
vocab_size = 32768
tokenizer = BPETokenizer(vocab_size=vocab_size)

print(f"Training BPE tokenizer with vocab size {vocab_size}...")

tokenizer = BPETokenizer(
    vocab_size=vocab_size,
    reserved_tokens=["<pad>", "<unk>", "<s>", "</s>", "<mask>", "<sep>", "<cls>"],
)
tokenizer.train(
    train_val_texts,
    save_checkpoints=[8192, 16384, 32768],
    checkpoint_dir=cache_dir
)

print("Tokenizer training finished!")

bpe_en_path = os.path.join(cache_dir, 'final_bpe_en.json')
tokenizer.save(bpe_en_path)


Training BPE tokenizer with vocab size 32768...

STEP 1: word frequencies
  [warn] example 281 has 548,028 chars -> truncated to 500,000
  [warn] example 284 has 706,211 chars -> truncated to 500,000
  [warn] example 363 has 715,208 chars -> truncated to 500,000
  [warn] example 375 has 1,225,790 chars -> truncated to 500,000
  [warn] example 490 has 1,114,127 chars -> truncated to 500,000
  [warn] example 531 has 760,892 chars -> truncated to 500,000
  [warn] example 592 has 1,641,907 chars -> truncated to 500,000
  [warn] example 601 has 574,721 chars -> truncated to 500,000
  [warn] example 670 has 539,918 chars -> truncated to 500,000
  [warn] example 954 has 688,873 chars -> truncated to 500,000
Examples: 1,000 | Unique words: 62,498 | RAM: 21.31 GB
  [warn] example 1,063 has 829,254 chars -> truncated to 500,000
  [warn] example 1,251 has 628,841 chars -> truncated to 500,000
  [warn] example 1,439 has 911,301 chars -> truncated to 500,000
  [warn] example 1,488 has 725,807 chars

In [ ]:
import torch
import os

def get_device():
    """Returns the available computing device (CUDA, MPS, or CPU)."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def save_checkpoint(model, optimizer, epoch, loss, filepath, best_loss=float('inf'), loss_history=None, global_step=0):
    """Saves the model and optimizer state to a checkpoint file."""
    os.makedirs(os.path.dirname(filepath) or '.', exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer else None,
        'loss': loss,
        'best_loss': best_loss,
        'loss_history': loss_history if loss_history is not None else [],
        'step': global_step, # Save global_step
    }
    torch.save(checkpoint, filepath)
    print(f"Checkpoint saved to {filepath}")

def load_checkpoint(filepath, model, optimizer=None):
    """Loads a model and optimizer from a checkpoint file."""
    if not os.path.exists(filepath):
        print(f"No checkpoint found at {filepath}")
        return 0, float('inf'), float('inf'), [], 0 # Added global_step as 0

    checkpoint = torch.load(filepath, map_location=get_device())
    model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer and checkpoint.get('optimizer_state_dict'):
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    epoch = checkpoint.get('epoch', 0)
    loss = checkpoint.get('loss', float('inf'))
    best_loss = checkpoint.get('best_loss', float('inf'))
    loss_history = checkpoint.get('loss_history', [])
    global_step = checkpoint.get('step', 0) # Load global_step

    print(f"Loaded checkpoint from '{filepath}' (Epoch {epoch}, Step {global_step})")

    return epoch, loss, best_loss, loss_history, global_step # Return global_step


In [ ]:
device = get_device()
print("Using device: ", device)

Using device:  cuda


In [ ]:
# ============================================================
# BPE TOKENIZER TEST
# ============================================================

_VOCAB_SIZE = 32768
bpe_en_path = os.path.join(cache_dir, f'final_bpe_en.json')
tokenizer = BPETokenizer.from_json(bpe_en_path)

print("Tokenizer loaded successfully")
print(f"Vocabulary size: {len(tokenizer.itos):,}")
print(f"Number of merges: {len(tokenizer.merges):,}")

test_texts = [
    "The Supreme Court of India held that the right to equality is a fundamental right.",

    "Section 302 of the Indian Penal Code provides punishment for murder.",

    "The petitioner challenged the constitutional validity of the impugned provision.",

    "Section 21: Protection of life and personal liberty shall not be denied.",
]

for text in test_texts:

    token_ids, token_strings = tokenizer.tokenize(
        text,
        add_special_tokens=True,
    )

    print("\n" + "=" * 80)
    print("TEXT:")
    print(text)

    print("\nTOKEN IDS:")
    print(token_ids)

    print("\nTOKENS:")
    print(token_strings)

    print("\nSTATISTICS:")
    print(f"Characters : {len(text):,}")
    print(f"Tokens     : {len(token_ids):,}")
    print(
        f"Chars/token: "
        f"{len(text) / max(len(token_ids), 1):.2f}"
    )

Tokenizer loaded successfully
Vocabulary size: 32,768
Number of merges: 32,504

TEXT:
The Supreme Court of India held that the right to equality is a fundamental right.

TOKEN IDS:
[2, 371, 2152, 498, 278, 961, 1189, 351, 271, 1516, 293, 9512, 306, 304, 9602, 1516, 272, 3]

TOKENS:
['<s>', 'The</w>', 'Supreme</w>', 'Court</w>', 'of</w>', 'India</w>', 'held</w>', 'that</w>', 'the</w>', 'right</w>', 'to</w>', 'equality</w>', 'is</w>', 'a</w>', 'fundamental</w>', 'right</w>', '.</w>', '</s>']

STATISTICS:
Characters : 82
Tokens     : 18
Chars/token: 4.56

TEXT:
Section 302 of the Indian Penal Code provides punishment for murder.

TOKEN IDS:
[2, 594, 4735, 278, 271, 1602, 2917, 1344, 4798, 3301, 319, 7165, 272, 3]

TOKENS:
['<s>', 'Section</w>', '302</w>', 'of</w>', 'the</w>', 'Indian</w>', 'Penal</w>', 'Code</w>', 'provides</w>', 'punishment</w>', 'for</w>', 'murder</w>', '.</w>', '</s>']

STATISTICS:
Characters : 68
Tokens     : 14
Chars/token: 4.86

TEXT:
The petitioner challenged the c

In [ ]:
import random

rng = random.Random(42)

indices = list(range(len(train_val_texts)))
# rng.shuffle(indices)

val_size = int(
    0.05 * len(indices)
)

val_indices = indices[:val_size]
train_indices = indices[val_size:]

train_texts = [
    train_val_texts[i]
    for i in train_indices
]

val_texts = [
    train_val_texts[i]
    for i in val_indices
]

print("Train:", len(train_texts))
print("Val:", len(val_texts))

Train: 499996
Val: 26315


In [ ]:
import gc
import numpy as np

def memory_safe_encode_and_save(texts, tokenizer, save_path, chunk_size=50000):
    """
    Encodes text dataset in small chunks and saves directly to disk to prevent RAM crashes.
    """
    print(f"Starting memory-safe encoding for {len(texts):,} documents...")

    # We will write the list of token IDs to a JSON-lines or partition file to keep RAM low
    with open(save_path, 'w', encoding='utf-8') as f:
        for i in range(0, len(texts), chunk_size):
            chunk = texts[i:i + chunk_size]
            print(f"  Processing chunk {i//chunk_size + 1} ({i:,} to {min(i+chunk_size, len(texts)):,})...")

            # Encode the small chunk
            encoded_chunk = tokenizer.encode_batch(
                chunk,
                add_special_tokens=True,
                num_workers=4,
                chunk_size=32
            )

            # Write to disk immediately
            for tokens in encoded_chunk:
                f.write(json.dumps(tokens) + '\n')

            # Free memory
            del chunk, encoded_chunk
            gc.collect()

    print(f"Finished encoding! Saved to {save_path}")

In [ ]:
# Define file paths for tokenized data on Google Drive
train_tokens_path = os.path.join(cache_dir, "train_tokens.jsonl")
val_tokens_path = os.path.join(cache_dir, "val_tokens.jsonl")

# Run memory-safe encoding
memory_safe_encode_and_save(train_texts, tokenizer, train_tokens_path)
memory_safe_encode_and_save(val_texts, tokenizer, val_tokens_path)

Starting memory-safe encoding for 499,996 documents...
  Processing chunk 1 (0 to 50,000)...
  Processing chunk 2 (50,000 to 100,000)...
  Processing chunk 3 (100,000 to 150,000)...
  Processing chunk 4 (150,000 to 200,000)...
  Processing chunk 5 (200,000 to 250,000)...
  Processing chunk 6 (250,000 to 300,000)...
  Processing chunk 7 (300,000 to 350,000)...
  Processing chunk 8 (350,000 to 400,000)...
  Processing chunk 9 (400,000 to 450,000)...
  Processing chunk 10 (450,000 to 499,996)...
Finished encoding! Saved to /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/train_tokens.jsonl
Starting memory-safe encoding for 26,315 documents...
  Processing chunk 1 (0 to 26,315)...
Finished encoding! Saved to /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/val_tokens.jsonl


In [ ]:
# Load the tokens back from disk in a memory-light manner
def load_tokens_from_jsonl(path):
    tokens_list = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            tokens_list.append(json.loads(line))
    return tokens_list


# Define file paths for tokenized data on Google Drive
train_tokens_path = os.path.join(cache_dir, "train_tokens.jsonl")
val_tokens_path = os.path.join(cache_dir, "val_tokens.jsonl")

train_tokens = load_tokens_from_jsonl(train_tokens_path)
val_tokens = load_tokens_from_jsonl(val_tokens_path)
print(f"Loaded {len(train_tokens):,} train token sequences and {len(val_tokens):,} validation sequences.")

Loaded 499,996 train token sequences and 26,315 validation sequences.


## Corrected Transformer-XL + GQA implementation

This section replaces the earlier Transformer-XL cells. It keeps the Transformer-XL segment recurrence and relative positional attention, while using grouped-query attention for Q/K/V. The training and validation loops carry memory sequentially across segments.


In [ ]:
import math
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
class TransformerXLRelativePositionalEncoding(nn.Module):
    def __init__(self, d_model: int, base: float = 10000.0):
        super().__init__()
        if d_model % 2:
            raise ValueError("d_model must be even")
        self.d_model = d_model
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2, dtype=torch.float32) / d_model))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, query_length: int, key_length: int, device=None):
        if key_length <= 0:
            raise ValueError("key_length must be positive")
        if device is None:
            device = self.inv_freq.device
        pos_seq = torch.arange(
            key_length - 1, -1, -1,
            dtype=torch.float32, device=device
        )
        sinusoid_inp = torch.outer(pos_seq, self.inv_freq.to(device))
        return torch.cat([sinusoid_inp.sin(), sinusoid_inp.cos()], dim=-1)

class TransformerXLGQAAttention(nn.Module):
    def __init__(self, # Removed d_h, d_v
                 d_model,
                 num_query_heads,
                 num_kv_heads,
                 dropout=0.0):
        super().__init__()
        if d_model % num_query_heads != 0:
            raise ValueError("d_model must be divisible by num_query_heads")
        if num_query_heads % num_kv_heads != 0:
            raise ValueError("num_query_heads must be divisible by num_kv_heads")
        self.d_model = d_model

        # Removed self.d_h and self.d_v as they are implicitly d_model for input features
        # and output dimensions are determined by heads * head_dim

        self.num_query_heads = num_query_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_query_heads
        self.group_size = num_query_heads//num_kv_heads

        # Input dimension is d_model for all projections
        self.q_proj = nn.Linear(d_model,
                                num_query_heads * self.head_dim,
                                bias=False)
        self.k_proj = nn.Linear(d_model,
                                num_kv_heads * self.head_dim,
                                bias=False)
        self.v_proj = nn.Linear(d_model,
                                num_kv_heads * self.head_dim,
                                bias=False)

        self.r_proj = nn.Linear(d_model,
                                num_kv_heads * self.head_dim,
                                bias=False)

        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.u = nn.Parameter(torch.zeros(num_query_heads, self.head_dim))
        self.v = nn.Parameter(torch.zeros(num_query_heads, self.head_dim))

        nn.init.normal_(self.u, std=0.02)
        nn.init.normal_(self.v, std=0.02)
        self.dropout = nn.Dropout(dropout)

    def _q_heads(self,x):
        B,T,_=x.shape
        return x.view(B,T,self.num_query_heads,self.head_dim).transpose(1,2)

    def _kv_heads(self,x):
        B,T,_=x.shape
        return x.view(B,T,self.num_kv_heads,self.head_dim).transpose(1,2)

    def _expand_kv(self,x):
        B,H,T,D=x.shape
        return x.unsqueeze(2).expand(B,H,self.group_size,T,D).reshape(B,self.num_query_heads,T,D)

    @staticmethod
    def _rel_shift(x):
        B,H,Q,K=x.shape
        zero=torch.zeros((B,H,Q,1),dtype=x.dtype,device=x.device)
        x=torch.cat([zero,x],dim=-1)
        x=x.view(B,H,K+1,Q)
        x=x[:,:,1:].view(B,H,Q,K)
        return x

    @staticmethod
    def _causal_mask(query_length,key_length,device):
        memory_length=key_length-query_length
        return torch.triu(
            torch.ones((query_length,key_length),dtype=torch.bool,device=device),
            diagonal=memory_length+1,
        )

    def forward(self, x , memory=None, relative_position=None):
        B,Q,_=x.shape
        if memory is not None and memory.size(1)>0:
            kv_input = torch.cat([memory,x],dim=1)
        else:
            kv_input = x
        K = kv_input.size(1)

        q = self._q_heads(self.q_proj(x))
        k = self._expand_kv(self._kv_heads(self.k_proj(kv_input)))
        v = self._expand_kv(self._kv_heads(self.v_proj(kv_input)))

        # Transformer-XL relative attention:
        # AC = q^T k + u^T k
        ac = torch.einsum("bhqd, bhkd->bhqk",q, k)
        ac = ac + torch.einsum("hd,bhkd->bhk",self.u,k).unsqueeze(2)

        if relative_position is None:
            raise ValueError("relative_position is required for Transformer-XL attention")

        r = self._expand_kv(self._kv_heads(self.r_proj(relative_position).unsqueeze(0)))
        bd = torch.einsum("bhqd,bhkd->bhqk",q,r)
        bd = bd+torch.einsum("hd,bhkd->bhk",self.v,r).unsqueeze(2)
        bd = self._rel_shift(bd)

        scores = (ac + bd) / math.sqrt(self.head_dim)
        mask = self._causal_mask(Q, K, x.device)
        scores = scores.masked_fill(mask.view(1,1,Q,K), torch.finfo(scores.dtype).min)
        attn = F.softmax(scores,dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn,v).transpose(1,2).contiguous().view(B,Q,self.d_model)
        return self.out_proj(out)


In [ ]:
class TransformerXLBlockGQA(nn.Module):
    def __init__(self,d_model,num_query_heads,num_kv_heads,dropout=0.0,ffn_mult=4):
        super().__init__()
        self.attn=TransformerXLGQAAttention(d_model,num_query_heads,num_kv_heads,dropout)
        self.norm1=nn.LayerNorm(d_model)
        self.norm2=nn.LayerNorm(d_model)
        ffn_dim=ffn_mult*d_model
        self.ffn=nn.Sequential(
            nn.Linear(d_model,ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim,d_model),
            nn.Dropout(dropout),
        )
    def forward(self,x,memory,relative_position):
        x=self.norm1(x+self.attn(x,memory=memory,relative_position=relative_position))
        x=self.norm2(x+self.ffn(x))
        return x

class TransformerXLGQAModel(nn.Module):
    def __init__(self,vocab_size,d_model,num_layers,num_query_heads,num_kv_heads,memory_length,dropout=0.0,tie_embeddings=True):
        super().__init__()
        self.vocab_size=vocab_size
        self.d_model=d_model
        self.num_layers=num_layers
        self.memory_length=memory_length
        self.embedding=nn.Embedding(vocab_size,d_model)
        self.pos_encoding=TransformerXLRelativePositionalEncoding(d_model)
        self.blocks=nn.ModuleList([
            TransformerXLBlockGQA(d_model,num_query_heads,num_kv_heads,dropout)
            for _ in range(num_layers)
        ])
        self.final_norm=nn.LayerNorm(d_model)
        self.lm_head=nn.Linear(d_model,vocab_size,bias=False)
        if tie_embeddings:
            self.lm_head.weight=self.embedding.weight

    def _update_memory(self,old_memory,x):
        if self.memory_length<=0:
            return None
        x=x.detach()
        if old_memory is None:
            return x[:,-self.memory_length:].contiguous()
        return torch.cat([old_memory,x],dim=1)[:,-self.memory_length:].detach().contiguous()

    def forward(self,input_ids,memory=None,labels=None):
        x=self.embedding(input_ids)
        Q=x.size(1)
        if memory is None:
            memory=[None]*self.num_layers
        if len(memory)!=self.num_layers:
            raise ValueError(f"memory must contain {self.num_layers} entries")
        mem_len=0 if memory[0] is None else memory[0].size(1)
        K=mem_len+Q
        rel_pos=self.pos_encoding(Q,K,device=x.device)
        new_memory=[]
        for i,block in enumerate(self.blocks):
            x=block(x,memory[i],rel_pos)
            new_memory.append(self._update_memory(memory[i],x))
        x=self.final_norm(x)
        logits=self.lm_head(x)
        loss=None
        if labels is not None:
            loss=F.cross_entropy(logits.reshape(-1,self.vocab_size),labels.reshape(-1))
        return {"logits":logits,"loss":loss,"memory":new_memory}

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, segment_length=128, temperature=1.0, top_k=None, top_p=None, eos_token_id=None):
        self.eval()
        device = input_ids.device
        memory = [None] * self.num_layers
        generated = input_ids.clone()

        # Process prompt exactly once, in sequential segments.
        last_logits = None
        for start in range(0, generated.size(1), segment_length):
            segment = generated[:, start:start + segment_length]
            out = self(segment, memory=memory)
            memory = out["memory"]
            last_logits = out["logits"][:, -1, :]

        for _ in range(max_new_tokens):
            logits = last_logits / temperature if temperature != 1.0 else last_logits

            if top_k is not None: # Only apply top_k if top_p is not applied
                # Apply Top-K filtering
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')

            if top_p is not None and top_p < 1.0:
                # Apply Top-P (Nucleus) filtering
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

                # Remove tokens with cumulative probability above the threshold (inclusive of the token that pushes it over)
                sorted_indices_to_remove = cumulative_probs > top_p
                # Shift the indices to the right to keep the first token above the threshold
                if sorted_indices_to_remove.shape[1] > 1:
                    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0

                # Set to -inf all logits where cumulative probability is too high
                logits = logits.scatter(
                    dim=-1,
                    index=sorted_indices,
                    src=sorted_logits.masked_fill(sorted_indices_to_remove, float('-inf'))
                )

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)

            generated = torch.cat([generated, next_token], dim=1)
            if eos_token_id is not None and bool((next_token == eos_token_id).all()):
                break
            out = self(next_token, memory=memory)
            memory = out["memory"]
            last_logits = out["logits"][:, -1, :]
        return generated


In [ ]:
class TransformerXLStreamLoader:
    """
    Transformer-XL stream loader.

    Documents are kept separate.
    Each document is split into sequential segments.
    Documents are shuffled between epochs, but segments belonging
    to the same document remain sequential.

    This prevents Transformer-XL memory from accidentally crossing
    unrelated document boundaries.
    """

    def __init__(
        self,
        token_ids_list,
        batch_size,
        segment_length,
        drop_last=True,
        seed=42,
    ):
        if batch_size <= 0 or segment_length <= 0:
            raise ValueError(
                "batch_size and segment_length must be positive"
            )

        self.batch_size = batch_size
        self.segment_length = segment_length
        self.drop_last = drop_last
        self.seed = seed

        # --------------------------------------------------------
        # Keep documents separate
        # --------------------------------------------------------

        self.documents = []

        for seq in token_ids_list:

            if isinstance(seq, torch.Tensor):
                seq = seq.tolist()

            if len(seq) >= segment_length + 1:
                self.documents.append(
                    torch.tensor(
                        seq,
                        dtype=torch.long,
                    )
                )

        if len(self.documents) == 0:
            raise ValueError(
                "No document is long enough for one segment."
            )

        # Current epoch
        self.epoch = 0

        # Build ordering
        self.set_epoch(0)

    # ------------------------------------------------------------
    # Shuffle documents every epoch
    # ------------------------------------------------------------

    def set_epoch(self, epoch):

        self.epoch = epoch

        generator = torch.Generator()
        generator.manual_seed(self.seed + epoch)

        permutation = torch.randperm(
            len(self.documents),
            generator=generator,
        )

        self.document_order = permutation.tolist()

    # ------------------------------------------------------------
    # Number of batches
    # ------------------------------------------------------------

    def __len__(self):

        total_segments = 0

        for idx in self.document_order:

            doc_len = self.documents[idx].numel()

            # Number of input/label segments
            n_segments = (
                (doc_len - 1)
                // self.segment_length
            )

            total_segments += n_segments

        if self.drop_last:
            return total_segments // self.batch_size

        return (
            total_segments + self.batch_size - 1
        ) // self.batch_size

    # ------------------------------------------------------------
    # Iterator
    # ------------------------------------------------------------

    def __iter__(self):

        segments = []

        # --------------------------------------------------------
        # Create sequential segments from each document
        # --------------------------------------------------------

        for doc_idx in self.document_order:

            tokens = self.documents[doc_idx]

            doc_len = tokens.numel()

            n_segments = (
                (doc_len - 1)
                // self.segment_length
            )

            for i in range(n_segments):

                start = i * self.segment_length

                input_ids = tokens[
                    start:
                    start + self.segment_length
                ]

                labels = tokens[
                    start + 1:
                    start + self.segment_length + 1
                ]

                if (
                    input_ids.numel()
                    != self.segment_length
                    or labels.numel()
                    != self.segment_length
                ):
                    continue

                segments.append(
                    {
                        "input_ids": input_ids,
                        "labels": labels,
                    }
                )

        # --------------------------------------------------------
        # Batch the segments
        # --------------------------------------------------------

        for start in range(
            0,
            len(segments),
            self.batch_size,
        ):

            batch_segments = segments[
                start:
                start + self.batch_size
            ]

            if (
                len(batch_segments)
                < self.batch_size
                and self.drop_last
            ):
                break

            yield {
                "input_ids": torch.stack(
                    [
                        x["input_ids"]
                        for x in batch_segments
                    ]
                ),
                "labels": torch.stack(
                    [
                        x["labels"]
                        for x in batch_segments
                    ]
                ),
            }

In [ ]:
def save_transformer_xl_checkpoint(filepath,model,optimizer=None,scheduler=None,scaler=None,epoch=0,global_step=0,best_val_loss=float("inf"),loss_history=None):
    os.makedirs(os.path.dirname(filepath) or ".",exist_ok=True)
    torch.save({
        "epoch":epoch,
        "global_step":global_step,
        "model_state_dict":model.state_dict(),
        "optimizer_state_dict":optimizer.state_dict() if optimizer else None,
        "scheduler_state_dict":scheduler.state_dict() if scheduler else None,
        "scaler_state_dict":scaler.state_dict() if scaler and scaler.is_enabled() else None,
        "best_val_loss":best_val_loss,
        "loss_history":loss_history or [],
    },filepath)

def load_transformer_xl_checkpoint(filepath,model,optimizer=None,scheduler=None,scaler=None,device="cpu"):
    if not filepath or not os.path.exists(filepath):
        return 0,0,float("inf"),[]
    ckpt=torch.load(filepath,map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and ckpt.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if scaler is not None and scaler.is_enabled() and ckpt.get("scaler_state_dict") is not None:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    return ckpt.get("epoch",0),ckpt.get("global_step",0),ckpt.get("best_val_loss",float("inf")),ckpt.get("loss_history",[])

def _amp_context(device,use_amp):
    enabled=bool(use_amp and device.type=="cuda")
    return torch.autocast(device_type="cuda",dtype=torch.float16,enabled=enabled)

@torch.no_grad()
def validate_transformer_xl(model,dataloader,device=None,use_amp=True,max_val_batches=None):
    if device is None:
        device=next(model.parameters()).device
    was_training=model.training
    model.eval()
    memory=[None]*model.num_layers
    total_nll=0.0
    total_tokens=0
    for idx, batch in enumerate(dataloader):
        if max_val_batches is not None and idx >= max_val_batches:
            break
        input_ids=batch["input_ids"].to(device,non_blocking=True)
        labels=batch["labels"].to(device,non_blocking=True)
        with _amp_context(device,use_amp):
            outputs=model(input_ids,memory=memory)
            logits=outputs["logits"]
        memory=outputs["memory"]
        loss_sum=F.cross_entropy(logits.reshape(-1,model.vocab_size),labels.reshape(-1),reduction="sum")
        total_nll += float(loss_sum.item())
        total_tokens += labels.numel()
    if was_training:
        model.train()
    return total_nll/max(total_tokens,1)

def train_transformer_xl(
    model,dataloader,val_dataloader,optimizer,scheduler=None,device=None,
    epochs=1,max_steps=None,grad_accum_steps=1,grad_clip=1.0,
    max_batches_per_epoch=None,max_val_batches=20_000,
    checkpoint_dir=None,tag="transformer_xl_gqa",save_every=1000,log_every=100,
    use_amp=True,
):
    if device is None:
        device=next(model.parameters()).device
    if checkpoint_dir:
        os.makedirs(checkpoint_dir,exist_ok=True)
    amp_enabled=bool(use_amp and device.type=="cuda")
    scaler=torch.amp.GradScaler("cuda",enabled=amp_enabled)

    resume_path=os.path.join(checkpoint_dir,f"{tag}.pt") if checkpoint_dir else None
    start_epoch,global_step,best_val_loss,history=load_transformer_xl_checkpoint(
        resume_path,model,optimizer,scheduler,scaler,device
    ) if resume_path else (0,0,float("inf"),[])

    print(f"Starting/resuming Transformer-XL + GQA training from epoch={start_epoch}, step={global_step}")

    for epoch in range(start_epoch,epochs):
        model.train()

        if hasattr(dataloader, "set_epoch"):
            dataloader.set_epoch(epoch)

        memory = [None] * model.num_layers
        running_loss=0.0
        running_tokens=0
        epoch_loss_sum=0.0
        epoch_tokens=0
        epoch_start=time.time()
        optimizer.zero_grad(set_to_none=True)

        num_batches = 0

        for batch_idx, batch in enumerate(dataloader):
            if (
                max_batches_per_epoch is not None
                and num_batches >= max_batches_per_epoch
            ):
                print(
                    f"Reached "
                    f"max_batches_per_epoch="
                    f"{max_batches_per_epoch:,}",
                    flush=True,
                )
                break

            input_ids=batch["input_ids"].to(device,non_blocking=True)
            labels=batch["labels"].to(device,non_blocking=True)

            with _amp_context(device,use_amp):
                outputs=model(input_ids,memory=memory)
                memory=outputs["memory"]
                loss=F.cross_entropy(
                    outputs["logits"].reshape(-1,model.vocab_size),
                    labels.reshape(-1)
                )
            if amp_enabled:
                scaler.scale(loss / grad_accum_steps).backward()
            else:
                (loss / grad_accum_steps).backward()

            running_loss += float(loss.item())*labels.numel()
            running_tokens += labels.numel()
            epoch_loss_sum += float(loss.item())*labels.numel()
            epoch_tokens += labels.numel()
            num_batches += 1

            should_step=((batch_idx + 1) % grad_accum_steps == 0) or ((batch_idx + 1) == len(dataloader))
            if should_step:
                if amp_enabled:
                    scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                if amp_enabled:
                    scaler.step(optimizer); scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                if scheduler is not None:
                    scheduler.step()
                global_step += 1

                if global_step % log_every == 0:
                    avg=running_loss/max(running_tokens,1)
                    ppl=math.exp(min(avg,20))
                    print(f"step={global_step:,} | batch={batch_idx+1:,} | loss={avg:.4f} | ppl={ppl:.2f} | tokens={running_tokens/1e6:.2f}M | grad_norm={float(grad_norm):.3f}",flush=True)
                    running_loss=0.0
                    running_tokens=0

                if checkpoint_dir and save_every and global_step % save_every == 0:
                    save_transformer_xl_checkpoint(
                        resume_path,model,optimizer,scheduler,scaler,
                        epoch=epoch,global_step=global_step,best_val_loss=best_val_loss,loss_history=history
                    )

                if max_steps is not None and global_step>=max_steps:
                    if checkpoint_dir:
                        save_transformer_xl_checkpoint(
                            resume_path,model,optimizer,scheduler,scaler,
                            epoch=epoch,global_step=global_step,best_val_loss=best_val_loss,loss_history=history
                        )
                    return global_step

        train_loss=epoch_loss_sum/max(epoch_tokens,1)
        val_loss=validate_transformer_xl(model,val_dataloader,device,use_amp,max_val_batches=max_val_batches)
        history.append({"epoch":epoch+1,"train_loss":train_loss,"val_loss":val_loss})
        val_ppl=math.exp(min(val_loss,20))
        train_ppl=math.exp(min(train_loss,20))
        print(f"\nEpoch {epoch+1} complete | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | train_ppl={train_ppl:.2f} | val_ppl={val_ppl:.2f} | time={(time.time()-epoch_start)/60:.2f} min")

        if val_loss < best_val_loss:
            best_val_loss=val_loss
            if checkpoint_dir:
                best_path=os.path.join(checkpoint_dir,f"best_model_{tag}.pt")
                save_transformer_xl_checkpoint(
                    best_path,model,optimizer,scheduler,scaler,
                    epoch=epoch+1,global_step=global_step,best_val_loss=best_val_loss,loss_history=history
                )
                print(f"Saved best model: {best_path}")

        if checkpoint_dir:
            save_transformer_xl_checkpoint(
                resume_path,model,optimizer,scheduler,scaler,
                epoch=epoch+1,global_step=global_step,best_val_loss=best_val_loss,loss_history=history
            )
            print(f"Saved resume checkpoint: {resume_path}")
    return global_step

### Configuration and sequential loaders

Unlike the GQLA dataloader, the Transformer-XL loader must not shuffle individual segments. Each batch row is a continuous token stream so that the memory from segment *t* corresponds to the same stream at segment *t+1*.


In [ ]:

# ============================================================
# Transformer-XL + GQA configuration
# ============================================================

batch_size = 6
segment_length = 128
memory_length = 128

num_layers = 32
d_model = 512
num_query_heads = 64
num_kv_heads = 16

dropout = 0.1

epochs = 7
grad_accum_steps = 32
learning_rate = 1e-4
weight_decay = 0.01
grad_clip = 1.0

use_amp = True
save_every = 250
log_every = 50

transformer_xl_checkpoint_dir = os.path.join(
    cache_dir,
    "transformer_xl_gqa_checkpoints",
)
os.makedirs(transformer_xl_checkpoint_dir, exist_ok=True)

print("Building sequential Transformer-XL loaders...")
train_loader_xl = TransformerXLStreamLoader(
    train_tokens,
    batch_size=batch_size,
    segment_length=segment_length,
)

val_loader_xl = TransformerXLStreamLoader(
    val_tokens,
    batch_size=batch_size,
    segment_length=segment_length,
)

del train_tokens, val_tokens

vocab_size = len(tokenizer.itos)
print("Vocab size: ", vocab_size)



Building sequential Transformer-XL loaders...
Vocab size:  32768


In [ ]:
import gc
import torch

# Delete old objects if they exist
for name in [
    "model",
    "optimizer",
    "scheduler",
    "scaler",
    "loss",
    "outputs",
    "logits",
    "batch",
    "input_ids",
    "labels",
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print(
    f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB"
)
print(
    f"Reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB"
)

Allocated: 0.00 GB
Reserved: 0.00 GB


In [ ]:

model = TransformerXLGQAModel(
    vocab_size=vocab_size,
    d_model=d_model,
    num_layers=num_layers,
    num_query_heads=num_query_heads,
    num_kv_heads=num_kv_heads,
    memory_length=memory_length,
    dropout=dropout,
).to(device)

optimizer = AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)

print(
    f"Transformer-XL + GQA parameters: "
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)
print(f"Train steps/epoch: {len(train_loader_xl):,}")
print(f"Val steps/epoch:   {len(val_loader_xl):,}")
print(f"Device: {device}")

Transformer-XL + GQA parameters: 107,136,000
Train steps/epoch: 848,486
Val steps/epoch:   195,192
Device: cuda


In [ ]:

print("Starting Transformer-XL + GQA training...")

final_step_xl = train_transformer_xl(
    model=model,
    dataloader=train_loader_xl,
    val_dataloader=val_loader_xl,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    grad_accum_steps=grad_accum_steps,
    grad_clip=grad_clip,
    checkpoint_dir=transformer_xl_checkpoint_dir,
    max_batches_per_epoch=100_000,
    tag="transformer_xl_gqa-large",
    save_every=save_every,
    log_every=log_every,
    use_amp=use_amp,
)

print(f"Transformer-XL + GQA training finished at step {final_step_xl:,}")


Starting Transformer-XL + GQA training...
Starting/resuming Transformer-XL + GQA training from epoch=0, step=0
step=50 | batch=400 | loss=76.4353 | ppl=485165195.41 | tokens=0.82M | grad_norm=418.932
step=100 | batch=800 | loss=39.8124 | ppl=485165195.41 | tokens=0.82M | grad_norm=138.408
step=150 | batch=1,200 | loss=34.9673 | ppl=485165195.41 | tokens=0.82M | grad_norm=69.623
step=200 | batch=1,600 | loss=29.0146 | ppl=485165195.41 | tokens=0.82M | grad_norm=52.964
step=250 | batch=2,000 | loss=28.1164 | ppl=485165195.41 | tokens=0.82M | grad_norm=87.223
step=300 | batch=2,400 | loss=25.0988 | ppl=485165195.41 | tokens=0.82M | grad_norm=74.105
step=350 | batch=2,800 | loss=23.6037 | ppl=485165195.41 | tokens=0.82M | grad_norm=52.268
step=400 | batch=3,200 | loss=22.8219 | ppl=485165195.41 | tokens=0.82M | grad_norm=37.901
step=450 | batch=3,600 | loss=20.0850 | ppl=485165195.41 | tokens=0.82M | grad_norm=97.694
step=500 | batch=4,000 | loss=17.9790 | ppl=64295883.89 | tokens=0.82M | 

In [ ]:

print("Starting Transformer-XL + GQA training...")


final_step_xl = train_transformer_xl(
    model=model,
    dataloader=train_loader_xl,
    val_dataloader=val_loader_xl,
    optimizer=optimizer,
    device=device,
    epochs=8,
    grad_accum_steps=grad_accum_steps,
    grad_clip=grad_clip,
    checkpoint_dir=transformer_xl_checkpoint_dir,
    max_batches_per_epoch=None,
    tag="transformer_xl_gqa-small",
    max_steps=100_000,
    save_every=save_every,
    log_every=log_every,
    use_amp=use_amp,
)

print(f"Transformer-XL + GQA training finished at step {final_step_xl:,}")


Starting Transformer-XL + GQA training...
Starting/resuming Transformer-XL + GQA training from epoch=7, step=30000
step=30,050 | batch=400 | loss=3.1162 | ppl=22.56 | tokens=0.82M | grad_norm=2.331
step=30,100 | batch=800 | loss=3.0613 | ppl=21.35 | tokens=0.82M | grad_norm=3.009
step=30,150 | batch=1,200 | loss=3.0812 | ppl=21.78 | tokens=0.82M | grad_norm=3.072
step=30,200 | batch=1,600 | loss=3.0764 | ppl=21.68 | tokens=0.82M | grad_norm=3.182
step=30,250 | batch=2,000 | loss=3.0931 | ppl=22.05 | tokens=0.82M | grad_norm=2.893
step=30,300 | batch=2,400 | loss=3.0407 | ppl=20.92 | tokens=0.82M | grad_norm=2.957
step=30,350 | batch=2,800 | loss=2.9657 | ppl=19.41 | tokens=0.82M | grad_norm=3.849
step=30,400 | batch=3,200 | loss=3.0372 | ppl=20.85 | tokens=0.82M | grad_norm=3.719
step=30,450 | batch=3,600 | loss=3.0309 | ppl=20.72 | tokens=0.82M | grad_norm=2.497
step=30,500 | batch=4,000 | loss=3.0398 | ppl=20.90 | tokens=0.82M | grad_norm=3.378
step=30,550 | batch=4,400 | loss=2.9230

In [ ]:

print("Starting Transformer-XL + GQA training...")

final_step_xl = train_transformer_xl(
    model=model,
    dataloader=train_loader_xl,
    val_dataloader=val_loader_xl,
    optimizer=optimizer,
    device=device,
    epochs=5,
    grad_accum_steps=grad_accum_steps,
    grad_clip=grad_clip,
    checkpoint_dir=transformer_xl_checkpoint_dir,
    max_batches_per_epoch=100_000,
    tag="transformer_xl_gqa-large",
    save_every=save_every,
    log_every=log_every,
    use_amp=use_amp,
)

print(f"Transformer-XL + GQA training finished at step {final_step_xl:,}")


Starting Transformer-XL + GQA training...
Starting/resuming Transformer-XL + GQA training from epoch=4, step=17250
step=17,300 | batch=1,600 | loss=6.8281 | ppl=923.39 | tokens=1.23M | grad_norm=5.895
step=17,350 | batch=3,200 | loss=6.8502 | ppl=944.11 | tokens=1.23M | grad_norm=4.623
step=17,400 | batch=4,800 | loss=6.8663 | ppl=959.43 | tokens=1.23M | grad_norm=4.488
step=17,450 | batch=6,400 | loss=6.8840 | ppl=976.52 | tokens=1.23M | grad_norm=5.031
step=17,500 | batch=8,000 | loss=6.8079 | ppl=904.96 | tokens=1.23M | grad_norm=3.276
step=17,550 | batch=9,600 | loss=6.8724 | ppl=965.29 | tokens=1.23M | grad_norm=9.898
step=17,600 | batch=11,200 | loss=6.8752 | ppl=967.99 | tokens=1.23M | grad_norm=4.696
step=17,650 | batch=12,800 | loss=6.8680 | ppl=961.02 | tokens=1.23M | grad_norm=4.225
step=17,700 | batch=14,400 | loss=6.8400 | ppl=934.44 | tokens=1.23M | grad_norm=3.602
step=17,750 | batch=16,000 | loss=6.8444 | ppl=938.59 | tokens=1.23M | grad_norm=5.428
step=17,800 | batch=1

In [ ]:
# Immediately save the current model state to prevent losing progress after interrupting
manual_save_path = os.path.join(transformer_xl_checkpoint_dir, "transformer_xl_gqa-large.pt")

try:
    save_transformer_xl_checkpoint(
        filepath=manual_save_path,
        model=model,
        optimizer=optimizer,
        epoch=1,  # Or the current epoch if you know it
        global_step=6125, # Adjust to the last known logged step
        best_val_loss=float("inf"),
        loss_history=[]
    )
    print(f"Success! Model checkpoint manually saved to {manual_save_path}")
except Exception as e:
    print(f"Failed to save checkpoint: {e}")

Success! Model checkpoint manually saved to /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/transformer_xl_gqa_checkpoints/transformer_xl_gqa-large.pt


In [ ]:
test_file_path = hf_hub_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    filename="Phase_1_Pretraining/Test/test.json",
    repo_type="dataset",
    token=hf_token
)

print("Test file:", test_file_path)

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

Test file: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json


In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

Number of test examples: 1000


In [ ]:
#!/usr/bin/env python3
"""Local evaluator for Phase 1.

Usage:
  python evaluate_phase1.py --truth truth.json --prediction prediction.json

Requires: bert-score and torch for the official primary metric.
"""
import argparse, json, math, sys
from collections import Counter

try:
    import bert_score
except ImportError as exc:
    raise SystemExit("Missing dependency: bert-score. Install it from requirements.docx / requirements.txt.") from exc

class DuplicateJSONKeyError(ValueError): pass

def reject_constant(v): raise ValueError(f"Non-standard JSON numeric value '{v}' is not allowed.")

def pairs_no_duplicates(pairs):
    out = {}; dup = []
    for k, v in pairs:
        if k in out: dup.append(k)
        out[k] = v
    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")
    return out

def load_json(path):
    try:
        with open(path, encoding='utf-8') as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e

def validate(data, name):
    if not isinstance(data, dict) or not data: raise ValueError(f"{name} must be a non-empty JSON object mapping IDs to records.")
    for i, rec in data.items():
        if not isinstance(i, str) or not i.strip() or not isinstance(rec, dict) or 'text' not in rec or not isinstance(rec['text'], str) or not rec['text'].strip():
            raise ValueError(f"Invalid record for ID {i!r} in {name}; each record must contain non-empty string field 'text'.")

def lcs_length(a,b):
    if not a or not b: return 0
    if len(a) < len(b): a,b=b,a
    masks={}
    for j,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<j)
    state=0
    for t in b:
        m=masks.get(t,0); x=m|state; y=(state<<1)|1; state=x & ~(x-y)
    return state.bit_count() if hasattr(int,'bit_count') else bin(state).count('1')

def rouge_l_f1(p,r):
    pt=p.split(); rt=r.split()
    if not pt or not rt: return 0.0
    l=lcs_length(pt,rt)
    if not l: return 0.0
    pp=l/len(pt); rr=l/len(rt)
    return 0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)

def ngrams(t,n): return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()

def bleu4(preds, refs):
    order=4; clipped=[0]*order; totals=[0]*order; pl=rl=0
    for p,r in zip(preds,refs):
        pl+=len(p); rl+=len(r)
        for n in range(1,order+1):
            pn=ngrams(p,n); rn=ngrams(r,n); totals[n-1]+=sum(pn.values())
            for g,c in pn.items(): clipped[n-1]+=min(c,rn.get(g,0))
    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(order)]
    if any(x<=0 for x in prec) or pl==0: return 0.0
    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)
    return bp*math.exp(sum(math.log(x) for x in prec)/order)

def run_evaluation(truth_path, prediction_path):
    truth = load_json(truth_path)
    pred = load_json(prediction_path)
    validate(truth, 'truth')
    validate(pred, 'prediction')

    if set(truth.keys()) != set(pred.keys()):
        # Find missing and extra keys for better error message
        missing_keys = set(truth.keys()) - set(pred.keys())
        extra_keys = set(pred.keys()) - set(truth.keys())
        error_msg = f"ID mismatch: "
        if missing_keys: error_msg += f"missing={len(missing_keys)} (e.g., {list(missing_keys)[:5]}), "
        if extra_keys: error_msg += f"extra={len(extra_keys)} (e.g., {list(extra_keys)[:5]}), "
        raise ValueError(error_msg.strip(', '))

    # Ensure consistent ordering for predictions and references based on truth keys
    ids_ordered = list(truth.keys())

    ps = [pred[i]['text'].split() for i in ids_ordered]
    rs = [truth[i]['text'].split() for i in ids_ordered]
    pred_texts = [pred[i]['text'].strip() for i in ids_ordered]
    ref_texts = [truth[i]['text'].strip() for i in ids_ordered]

    try:
        import torch
        device='cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError:
        device='cpu'
        print("PyTorch not found or CUDA not available, using CPU for BERTScore.")

    print("Calculating BERTScore...")
    _,_,f = bert_score.score(pred_texts, ref_texts, lang='en', batch_size=8, verbose=False, **({'device':device} if device else {}))
    bertscore = float(f.mean().item())
    print(f"BERTScore: {bertscore:.4f}")

    print("Calculating ROUGE-L...")
    rouge = sum(rouge_l_f1(pred[i]['text'],truth[i]['text']) for i in ids_ordered)/len(ids_ordered)
    print(f"ROUGE-L: {rouge:.4f}")

    print("Calculating BLEU-4...")
    bleu = bleu4(ps, rs)
    print(f"BLEU-4: {bleu:.4f}")

    return {
        "bertscore": bertscore,
        "rouge_l": rouge,
        "bleu_4": bleu
    }

# The main function is removed as this script will be imported and its functions called directly.


### Load Best Model and Prepare for Prediction

First, I will load the best performing model based on validation loss from the saved checkpoint `best_model.pt`.

In [ ]:
import torch
import os
import gc

# Clear GPU memory before loading the model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# Define the path to the best model checkpoint
best_model_path = os.path.join(transformer_xl_checkpoint_dir, "best_model_transformer_xl_gqa-small.pt")

model = TransformerXLGQAModel(
    vocab_size=vocab_size,
    d_model=d_model,
    num_layers=num_layers,
    num_query_heads=num_query_heads,
    num_kv_heads=num_kv_heads,
    memory_length=memory_length,
    dropout=dropout,
).to(device)



# Load the best model
# We don't need the optimizer state for inference
_, _, _, _, _ = load_checkpoint(best_model_path, model)
model.eval()

print("Best model loaded successfully for prediction.")

Loaded checkpoint from '/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/transformer_xl_gqa_checkpoints/best_model_transformer_xl_gqa-small.pt' (Epoch 7, Step 0)
Best model loaded successfully for prediction.


In [ ]:
import json

file_name = "model_config-small-txl.json"

model_config = {
    "vocab_size": vocab_size,
    "d_model": d_model,
    "num_query_heads": num_query_heads,
    "num_kv_heads": num_kv_heads,
    "memory_length": memory_length,
    "dropout": dropout
}

with open(os.path.join(transformer_xl_checkpoint_dir, file_name), "w") as f:
  json.dump(model_config, f ,indent=4)

print("Model config saved...")

Model config saved...


### Generate Predictions and Ground Truth

Now, I will iterate through the `test_data`, tokenize each example, split it into a prompt and a target continuation, generate text based on the prompt using the loaded model, and then decode both the generated text and the true target continuation.

These will be stored in `predictions_output` and `truth_output` dictionaries, mapping the example ID to the generated/true text.

In [ ]:
CONTEXT_LEN = 512
TARGET_LEN = 128
TOTAL_LEN = CONTEXT_LEN + TARGET_LEN

# Initialize dictionaries to store predictions and ground truth
predictions_output = {}
truth_output = {}

print(f"Generating predictions for {len(test_data)} test examples...")

example_no = 1

for example_id, example_data in list(test_data.items()):
    print(f"Processing example {example_no}/{len(test_data)}: {example_id}")
    example_no += 1
    # same normalisation as training (whitespace / quotes / invisible chars)
    input_text = normalize_text(example_data['text'])

    # Tokenize the entire input text without pre-padding
    original_token_ids = tokenizer.encode(
        input_text,
        add_special_tokens=True
    )

    # Determine prompt and true target token IDs based on actual content
    prompt_ids = original_token_ids[:CONTEXT_LEN]
    true_target_ids = original_token_ids[CONTEXT_LEN:CONTEXT_LEN + TARGET_LEN]

    # Decode true target text first to check if it's empty
    true_text = tokenizer.decode(true_target_ids)

    # Skip examples where the true target text is empty after decoding
    if not true_text.strip():
        print(f"  Skipping example {example_id} due to empty true target text.")
        continue

    # Pad prompt_ids to CONTEXT_LEN if necessary for model.generate, or simply use as is
    # If model.generate can handle variable length input_ids, no explicit padding is needed here for generation
    # For simplicity, we'll pass the exact prompt_ids length, assuming generate handles it.
    prompt_tensor = torch.tensor(
        prompt_ids,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    generated_ids = model.generate(
        input_ids=prompt_tensor,
        max_new_tokens=TARGET_LEN,
        temperature=1, # Use temperature=1.0 for nucleus sampling, or adjust as needed
        top_k=None,      # Disable top_k when using top_p for clearer precedence
        top_p=0.85,       # Enable Top-p (Nucleus) sampling
        eos_token_id=tokenizer.stoi['</s>']
    )

    # Extract generated continuation IDs, adjusting for potentially shorter generated sequences
    # generated_ids includes the prompt, so we take from after the prompt length
    start_idx = generated_ids.size(1) - TARGET_LEN
    generated_continuation_ids = generated_ids[
        0, start_idx:
    ].tolist()

    predicted_text = tokenizer.decode(generated_continuation_ids)

    # Store only if both predicted and true texts are non-empty for meaningful evaluation
    if predicted_text.strip() and true_text.strip():
        predictions_output[example_id] = {'text': predicted_text}
        truth_output[example_id] = {'text': true_text}
    else:
        print(f"  Skipping example {example_id} because predicted or true text is empty after decoding.")




print(f"Generated predictions for {len(predictions_output)} examples (after filtering for non-empty truth).")



Generating predictions for 1000 test examples...
Processing example 1/1000: 1
Processing example 2/1000: 2
Processing example 3/1000: 3
Processing example 4/1000: 4
Processing example 5/1000: 5
Processing example 6/1000: 6
Processing example 7/1000: 7
Processing example 8/1000: 8
Processing example 9/1000: 9
Processing example 10/1000: 10
Processing example 11/1000: 11
Processing example 12/1000: 12
Processing example 13/1000: 13
Processing example 14/1000: 14
Processing example 15/1000: 15
Processing example 16/1000: 16
Processing example 17/1000: 17
Processing example 18/1000: 18
Processing example 19/1000: 19
Processing example 20/1000: 20
Processing example 21/1000: 21
Processing example 22/1000: 22
Processing example 23/1000: 23
Processing example 24/1000: 24
Processing example 25/1000: 25
Processing example 26/1000: 26
Processing example 27/1000: 27
Processing example 28/1000: 28
Processing example 29/1000: 29
Processing example 30/1000: 30
Processing example 31/1000: 31
Processi

In [ ]:
true_text

'to deprive the rights of the others in relation to a property would render the transaction void ab initio . Fraud and deception are synonymous . Although in a given case a deception may not amount to fraud , fraud is anathema to all equitable principles and any affair tainted with fraud cannot be perpetuated or saved by the application of any equitable doctrine including res judicata . Fraud is proved when it is shown that a false representation has been made ( i ) knowingly , or ( ii ) without belief in its truth , or ( iii ) recklessly , careless whether it be true or false . Suppression'

In [ ]:
predicted_text

'to arbitration and equal and Office Memorandum on its orders and of lands in their house for his family shall be bearing registration . . . . In view of the past service is less than that he submits that but also punishable under Section 69 shall not apply and after unless they are established under [ section 42 ] [ The assessee was sought certain information regarding a demand or information in manner which contains sales , and without recovery in respect of Kumari and pertains to consumers or remaining HQ or has sought a suit in a suit for recovery of power under Section 2 ( 4 ) ? " zpartition " is delivered or made the rights of a relief is expressed to section'

### Save Predictions and Ground Truth to JSON Files

To evaluate the model, I will save the generated predictions and the corresponding ground truth continuations into `prediction.json` and `truth.json` files, respectively. These files will be used by the evaluation script.

In [ ]:
import json

prediction_json_path = os.path.join(cache_dir, "prediction.json")
truth_json_path = os.path.join(cache_dir, "truth.json")

with open(prediction_json_path, "w", encoding="utf-8") as f:
    json.dump(predictions_output, f, ensure_ascii=False, indent=2)
print(f"Predictions saved to: {prediction_json_path}")

with open(truth_json_path, "w", encoding="utf-8") as f:
    json.dump(truth_output, f, ensure_ascii=False, indent=2)
print(f"Ground truth saved to: {truth_json_path}")

Predictions saved to: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/prediction.json
Ground truth saved to: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/truth.json


### Run Local Evaluation Script

Finally, I will run the `run_evaluation` function using the `truth.json` and `prediction.json` files to compute BERTScore, ROUGE-L, and BLEU-4 metrics.

In [ ]:
evaluation_results = run_evaluation(truth_json_path, prediction_json_path)

print("\nEvaluation Results:")
for metric, score in evaluation_results.items():
    print(f"- {metric.replace('_', '-').upper()}: {score:.4f}")

Calculating BERTScore...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore: 0.7872
Calculating ROUGE-L...
ROUGE-L: 0.1346
Calculating BLEU-4...
BLEU-4: 0.0094

Evaluation Results:
- BERTSCORE: 0.7872
- ROUGE-L: 0.1346
- BLEU-4: 0.0094


In [ ]:
import itertools
import pandas as pd
import numpy as np

def run_parameter_experiment(model, tokenizer, test_data, temp_range, top_p_range, top_k_range, max_eval_examples=100, eval_samples=3):
    """
    Runs generation evaluations across a grid of hyperparameters on a subset of the test data.

    Args:
        model: The trained TransformerXLGQAModel.
        tokenizer: The BPETokenizer.
        test_data: Dict of test examples.
        temp_range: List of temperatures to test, e.g., [0.7, 1.0]
        top_p_range: List of top_p values to test, e.g., [0.8, 0.9, None]
        top_k_range: List of top_k values to test, e.g., [None, 50]
        max_eval_examples: Limits the number of evaluation examples per run to keep it fast.
    """

    results = []

    # Generate parameter combinations
    combinations = list(itertools.product(temp_range, top_p_range, top_k_range))
    print(f"Starting grid search over {len(combinations)} hyperparameter combinations...")

    # Temporary file paths
    temp_pred_path = os.path.join(cache_dir, "temp_eval_pred.json")
    temp_truth_path = os.path.join(cache_dir, "temp_eval_truth.json")


    for temp, top_p, top_k in combinations:
        print(f"\nTesting Configuration -> Temp: {temp}, Top-P: {top_p}, Top-K: {top_k}")

        config_results = {
            "bertscores": [],
            "bleu_4": [],
            "rouge_l": []
        }

        for _ in range(eval_samples):

            test_subset = dict(list(test_data.items())[:max_eval_examples])
            preds = {}
            truth = {}

            for example_id, example_data in test_subset.items():
                # same normalisation as training (whitespace / quotes / invisible chars)
                input_text = normalize_text(example_data['text'])
                original_token_ids = tokenizer.encode(input_text, add_special_tokens=True)

                prompt_ids = original_token_ids[:CONTEXT_LEN]
                true_target_ids = original_token_ids[CONTEXT_LEN:CONTEXT_LEN + TARGET_LEN]

                true_text = tokenizer.decode(true_target_ids)
                if not true_text.strip():
                    continue

                prompt_tensor = torch.tensor(prompt_ids, dtype=torch.long).unsqueeze(0).to(device)

                generated_ids = model.generate(
                    input_ids=prompt_tensor,
                    max_new_tokens=TARGET_LEN,
                    temperature=temp,
                    top_p=top_p,
                    top_k=top_k,
                    eos_token_id=tokenizer.stoi['</s>']
                )

                start_idx = generated_ids.size(1) - TARGET_LEN
                gen_ids = generated_ids[0, start_idx:].tolist()
                predicted_text = tokenizer.decode(gen_ids)

                if predicted_text.strip() and true_text.strip():
                    preds[example_id] = {'text': predicted_text}
                    truth[example_id] = {'text': true_text}

            # Save temporary files to evaluate via your custom run_evaluation script
            with open(temp_pred_path, "w", encoding="utf-8") as f:
                json.dump(preds, f, ensure_ascii=False)
            with open(temp_truth_path, "w", encoding="utf-8") as f:
                json.dump(truth, f, ensure_ascii=False)

            # Compute metrics
            try:
                metrics = run_evaluation(temp_truth_path, temp_pred_path)
                config_results['bertscores'].append(metrics["bertscore"])
                config_results['rouge_l'].append(metrics['rouge_l'])
                config_results['bleu_4'].append(metrics['bleu_4'])

            except Exception as e:
                print(f"Evaluation failed for current config: {e}")

        results.append({
                  "temperature": temp,
                  "top_p": top_p,
                  "top_k": top_k,
                  "bertscore": np.mean(config_results['bertscores']),
                  "rouge_l": np.mean(config_results['rouge_l']),
                  "bleu_4": np.mean(config_results['bleu_4']),
                  'results': config_results
              })

    # Return sorted results

    return results

In [ ]:
# Define your search spaces here
temperatures = [t * 0.1 for t in range(5, 16)]
top_p_values = [None] + [p * 0.01 for p in range(65, 100, 10)]
top_k_values = [None,  25, 50, 100]

CONTEXT_LEN = 512
TARGET_LEN = 128
TOTAL_LEN = CONTEXT_LEN + TARGET_LEN

# Run the experiment over a fast subset of 50 documents to verify performance
results = run_parameter_experiment(
    model=model,
    tokenizer=tokenizer,
    test_data=test_data,
    temp_range=temperatures,
    top_p_range=top_p_values,
    top_k_range=top_k_values,
    max_eval_examples=50
)

results_df = pd.DataFrame(results)
results_df.sort_values(by="bertscore", ascending=False)

display(results_df)

In [ ]:
results_df.to_csv(
    os.path.join(transformer_xl_checkpoint_dir,
                 'results.csv'),
    index=False
    )